In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from keras.callbacks import EarlyStopping,ReduceLROnPlateau
from matplotlib import pyplot as plt

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"keshrivishal21","key":"4c8bb86460f4dd386e694fd83bc26447"}'}

In [4]:
!mkdir -p ~/.config/kaggle
!mv kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

In [5]:
!kaggle competitions download -c state-farm-distracted-driver-detection

100% 4.00G/4.00G [04:39<00:00, 15.4MB/s]



In [6]:
!unzip -q state-farm-distracted-driver-detection.zip -d data

In [7]:
df = pd.read_csv("/content/data/driver_imgs_list.csv")
df.head()

,subject,classname,img
0,p002,c0,img_44733.jpg
1,p002,c0,img_72999.jpg
2,p002,c0,img_25094.jpg
3,p002,c0,img_69092.jpg
4,p002,c0,img_92629.jpg


In [8]:
drivers = df['subject'].unique()
print("Total drivers:", len(drivers))

Total drivers: 26


In [9]:
train_drivers, temp_drivers = train_test_split(
    drivers,
    test_size=0.4,
    random_state=42
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.5,
    random_state=42
)

print("Train drivers:", len(train_drivers))
print("Val drivers:", len(val_drivers))
print("Test drivers:", len(test_drivers))

Train drivers: 15
Val drivers: 5
Test drivers: 6


In [10]:
train_df = df[df['subject'].isin(train_drivers)]
val_df = df[df['subject'].isin(val_drivers)]
test_df = df[df['subject'].isin(test_drivers)]

print(train_df['subject'].nunique())
print(val_df['subject'].nunique())
print(test_df['subject'].nunique())

15
5
6


In [11]:
base_path = "/content/data/imgs/train"

def add_image_path(df):
    return df.apply(
        lambda row: os.path.join(base_path, row['classname'], row['img']),
        axis=1
    )

train_paths = add_image_path(train_df)
val_paths = add_image_path(val_df)
test_paths = add_image_path(test_df)

train_labels = train_df['classname']
val_labels = val_df['classname']
test_labels = test_df['classname']

In [12]:
le = LabelEncoder()
train_labels = le.fit_transform(train_labels)
val_labels = le.transform(val_labels)
test_labels = le.transform(test_labels)

In [13]:
IMG_SIZE = (224,224)
BATCH_SIZE = 16
tf.random.set_seed(42)

def process_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3, try_recover_truncated=True)

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]

    top = tf.cast(0.25 * tf.cast(h, tf.float32), tf.int32)
    bottom = tf.cast(0.95 * tf.cast(h, tf.float32), tf.int32)
    left = tf.cast(0.15 * tf.cast(w, tf.float32), tf.int32)
    right = tf.cast(0.85 * tf.cast(w, tf.float32), tf.int32)

    image = image[top:bottom, left:right]

    image = tf.image.resize(image, IMG_SIZE, method='bilinear', antialias=True)
    image = tf.cast(image, tf.float32)

    return image, label

def create_dataset(paths, labels, shuffle=True):
    labels = tf.convert_to_tensor(labels, dtype=tf.int32)
    labels = tf.one_hot(labels, depth=10)

    paths = paths.values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=1500, reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE, drop_remainder=True)

    ds = ds.prefetch(tf.data.AUTOTUNE)

    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    return ds


train_ds = create_dataset(train_paths, train_labels, shuffle=True)
val_ds = create_dataset(val_paths, val_labels, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, shuffle=False)

In [14]:
for path in train_paths[:5]:
    print(path, os.path.exists(path))

/content/data/imgs/train/c0/img_48693.jpg True
/content/data/imgs/train/c0/img_44903.jpg True
/content/data/imgs/train/c0/img_58514.jpg True
/content/data/imgs/train/c0/img_62307.jpg True
/content/data/imgs/train/c0/img_83984.jpg True


In [15]:
set(train_drivers) & set(val_drivers)
set(train_drivers) & set(test_drivers)
set(val_drivers) & set(test_drivers)

set()

In [16]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

Image batch shape: (16, 224, 224, 3)
Label batch shape: (16, 10)


In [17]:
print("Train images:", len(train_paths))
print("Val images:", len(val_paths))
print("Test images:", len(test_paths))

Train images: 13440
Val images: 3312
Test images: 5672


In [17]:
def mobilenetv3_small_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

    # Base Model
    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # Classification Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [ ]:
model, base_model = mobilenetv3_small_model()
model.summary()

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,017,082 (3.88 MB)

 Trainable params: 76,554 (299.04 KB)

 Non-trainable params: 940,528 (3.59 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 99s 74ms/step - accuracy: 0.2156 - loss: 2.5923 - val_accuracy: 0.3083 - val_loss: 2.3056 - learning_rate: 1.0000e-04
Epoch 2/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 86s 72ms/step - accuracy: 0.4138 - loss: 1.8970 - val_accuracy: 0.3693 - val_loss: 2.1121 - learning_rate: 1.0000e-04
Epoch 3/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 83s 69ms/step - accuracy: 0.5164 - loss: 1.6452 - val_accuracy: 0.3684 - val_loss: 2.0670 - learning_rate: 1.0000e-04
Epoch 4/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 88s 74ms/step - accuracy: 0.5518 - loss: 1.5366 - val_accuracy: 0.3979 - val_loss: 1.9993 - learning_rate: 1.0000e-04
Epoch 5/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 81s 68ms/step - accuracy: 0.6106 - loss: 1.4290 - val_accuracy: 0.4004 - val_loss: 2.0106 - learning_rate: 1.0000e-04
Epoch 6/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 76s 63ms/step - accuracy: 0.6370 - loss: 1.3521 - val_accuracy: 0.4146 - val_loss: 1.9855 - learning_rate: 1.0000e-04
Epoch 7/25
1120/1120 ━━━━━━━━━━━━━━━━━━━

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

472/472 ━━━━━━━━━━━━━━━━━━━━ 23s 49ms/step - accuracy: 0.4811 - loss: 1.7326
Driver-wise Test Accuracy: 0.48110875487327576


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,170,192 (4.46 MB)

 Trainable params: 835,642 (3.19 MB)

 Non-trainable params: 181,440 (708.75 KB)

 Optimizer params: 153,110 (598.09 KB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "mobileNetV3_small_model.keras",
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 1/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 101s 74ms/step - accuracy: 0.5417 - loss: 1.4559 - val_accuracy: 0.4469 - val_loss: 1.9511 - learning_rate: 3.0000e-06
Epoch 2/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 93s 79ms/step - accuracy: 0.5966 - loss: 1.3294 - val_accuracy: 0.4472 - val_loss: 1.8318 - learning_rate: 3.0000e-06
Epoch 3/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 95s 79ms/step - accuracy: 0.6244 - loss: 1.2542 - val_accuracy: 0.4626 - val_loss: 1.7803 - learning_rate: 3.0000e-06
Epoch 4/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 86s 73ms/step - accuracy: 0.6558 - loss: 1.1746 - val_accuracy: 0.4780 - val_loss: 1.7536 - learning_rate: 3.0000e-06
Epoch 5/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 85s 72ms/step - accuracy: 0.6763 - loss: 1.1208 - val_accuracy: 0.4891 - val_loss: 1.7216 - learning_rate: 3.0000e-06
Epoch 6/25
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 143s 72ms/step - accuracy: 0.7051 - loss: 1.0661 - val_accuracy: 0.4982 - val_loss: 1.7046 - learning_rate: 3.0000e-06
Epoch 7/25
1120/1120 ━━━━━━━━━━━━━━━━━

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

In [18]:
def mobilenetv3_large_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV3Large(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [19]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [20]:
model, base_model = mobilenetv3_large_model()
model.summary()

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       123,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,125,002 (11.92 MB)

 Trainable params: 126,474 (494.04 KB)

 Non-trainable params: 2,998,528 (11.44 MB)

In [21]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 84ms/step - accuracy: 0.2841 - loss: 2.3737 - val_accuracy: 0.3152 - val_loss: 2.3352 - learning_rate: 1.0000e-04
Epoch 2/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 74s 82ms/step - accuracy: 0.5117 - loss: 1.6654 - val_accuracy: 0.3717 - val_loss: 2.3351 - learning_rate: 1.0000e-04
Epoch 3/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 80s 89ms/step - accuracy: 0.6136 - loss: 1.4257 - val_accuracy: 0.3940 - val_loss: 2.1662 - learning_rate: 1.0000e-04
Epoch 4/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 82s 91ms/step - accuracy: 0.6728 - loss: 1.2974 - val_accuracy: 0.4221 - val_loss: 2.1121 - learning_rate: 1.0000e-04
Epoch 5/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 73s 81ms/step - accuracy: 0.7080 - loss: 1.2235 - val_accuracy: 0.4266 - val_loss: 2.1412 - learning_rate: 1.0000e-04
Epoch 6/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 84s 82ms/step - accuracy: 0.7374 - loss: 1.1606 - val_accuracy: 0.4423 - val_loss: 2.0123 - learning_rate: 1.0000e-04
Epoch 7/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 81s 90ms/ste

In [22]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 24s 67ms/step - accuracy: 0.4666 - loss: 1.7236
Driver-wise Test Accuracy: 0.46663135290145874


In [23]:
base_model.trainable = True

for layer in base_model.layers[:-80]:
    layer.trainable = False

In [28]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       123,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,377,952 (12.89 MB)

 Trainable params: 2,921,362 (11.14 MB)

 Non-trainable params: 203,640 (795.47 KB)

 Optimizer params: 252,950 (988.09 KB)

In [24]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "mobileNetV3_large_model.keras",
    monitor='val_loss',
    save_best_only=True
)

In [25]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 1/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 104s 99ms/step - accuracy: 0.5874 - loss: 1.3467 - val_accuracy: 0.4976 - val_loss: 1.8207 - learning_rate: 3.0000e-06
Epoch 2/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 86s 97ms/step - accuracy: 0.6515 - loss: 1.1792 - val_accuracy: 0.5254 - val_loss: 1.6348 - learning_rate: 3.0000e-06
Epoch 3/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 98ms/step - accuracy: 0.7017 - loss: 1.0618 - val_accuracy: 0.5414 - val_loss: 1.5798 - learning_rate: 3.0000e-06
Epoch 4/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.7492 - loss: 0.9501 - val_accuracy: 0.5565 - val_loss: 1.5341 - learning_rate: 3.0000e-06
Epoch 5/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 97ms/step - accuracy: 0.7817 - loss: 0.8861 - val_accuracy: 0.5628 - val_loss: 1.5079 - learning_rate: 3.0000e-06
Epoch 6/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 96s 107ms/step - accuracy: 0.8019 - loss: 0.8307 - val_accuracy: 0.5707 - val_loss: 1.4764 - learning_rate: 3.0000e-06
Epoch 7/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 133s 99ms/

In [26]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 23s 66ms/step - accuracy: 0.6841 - loss: 1.1831
Driver-wise Test Accuracy: 0.684145450592041
